## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

In [1]:
!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00


## 1. Import Libraries

In [2]:
import os
import re
import gc
import math
import json
import random
import unicodedata
import fitz  # PyMuPDF
import torch
import inspect
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, set_seed,
)
from peft import (
    LoraConfig, TaskType, get_peft_model,
    prepare_model_for_kbit_training, PeftModel,
)
from google.colab import userdata
from huggingface_hub import HfApi

In [3]:
WRITE_TOKEN = userdata.get('HF_TOKEN_WRITE')

api = HfApi(token=WRITE_TOKEN)

## 2. Global Configurations

In [4]:
@dataclass
class Config:
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Output directories.
    output_dir: str = "/content/pharma_tinyllama_lora_output"
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"
    processed_data_dir: str = "/content/pharma_processed_data"

    # Text preprocessing.
    min_chars_per_paragraph: int = 80
    block_size: int = 512

    # Train/eval split.
    test_size: float = 0.15
    seed: int = 42

    # LoRA parameters.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training parameters.
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 10
    save_steps: int = 25
    save_total_limit: int = 2

    # For a quick demo, set max_steps to 20 or 30.
    # For a complete run, keep it as -1.
    max_steps: int = -1

In [5]:
config = Config()
config

Config(pdf_path='/content/Metformin-Lipid-Therapy-Knowledge.pdf', model_name='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', output_dir='/content/pharma_tinyllama_lora_output', adapter_dir='/content/pharma_tinyllama_lora_adapter', processed_data_dir='/content/pharma_processed_data', min_chars_per_paragraph=80, block_size=512, test_size=0.15, seed=42, lora_r=16, lora_alpha=32, lora_dropout=0.05, num_train_epochs=3.0, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, learning_rate=0.0002, warmup_ratio=0.03, weight_decay=0.01, logging_steps=5, eval_steps=10, save_steps=25, save_total_limit=2, max_steps=-1)

In [6]:
config.output_dir

'/content/pharma_tinyllama_lora_output'

In [7]:
config.processed_data_dir

'/content/pharma_processed_data'

In [8]:
print(json.dumps(asdict(config), indent=2))

{
  "pdf_path": "/content/Metformin-Lipid-Therapy-Knowledge.pdf",
  "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
  "output_dir": "/content/pharma_tinyllama_lora_output",
  "adapter_dir": "/content/pharma_tinyllama_lora_adapter",
  "processed_data_dir": "/content/pharma_processed_data",
  "min_chars_per_paragraph": 80,
  "block_size": 512,
  "test_size": 0.15,
  "seed": 42,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "num_train_epochs": 3.0,
  "per_device_train_batch_size": 1,
  "per_device_eval_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.01,
  "logging_steps": 5,
  "eval_steps": 10,
  "save_steps": 25,
  "save_total_limit": 2,
  "max_steps": -1
}


In [9]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

## 3. Check PDF

In [10]:
if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

PDF found: /content/Metformin-Lipid-Therapy-Knowledge.pdf


## 4. Extract Text from PDF

In [11]:
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [12]:
config.pdf_path

'/content/Metformin-Lipid-Therapy-Knowledge.pdf'

In [13]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [14]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

Total pages with extracted text: 6
Page-level character counts:
Page 1: 2244 characters
Page 2: 2889 characters
Page 3: 2636 characters
Page 4: 2416 characters
Page 5: 2613 characters
Page 6: 2761 characters


In [15]:
print(pdf_pages[0]["text"])

Metformin is one of the most widely prescribed oral antihyperglycemic agents.​
 Its primary mechanism of action involves the activation of AMP-activated protein kinase 
(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation 
while inhibiting hepatic gluconeogenesis.​
 Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes 
and display anti-inflammatory properties.​
 Recent studies also suggest potential anticancer effects through inhibition of the mTOR 
signaling pathway and suppression of tumor angiogenesis. 
 
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in 
significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to 
monotherapy.​
 Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal 
wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, 
suppressing endogenous cho

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


## 5. Text Cleaning Utilities

In [25]:
def clean_pdf_text(text: str) -> str:
    # Standardize Unicode text so visually similar characters are treated consistently.
    # Example: "ＡＭＰＫ" becomes "AMPK" and "ﬁ" becomes "fi".
    text = unicodedata.normalize("NFKC", text)

    # Remove invisible characters that may appear during PDF text extraction.
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # Join words broken by line hyphenation, e.g., "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Replace multiple spaces/tabs with a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Convert three or more newlines into a standard paragraph break.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove lines that contain only page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Split text into paragraphs, clean each paragraph, and remove empty ones.
    paragraphs = []
    for paragraph in re.split(r"\n\s*\n", text):
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()

        if paragraph:
            paragraphs.append(paragraph)

    # Join cleaned paragraphs with one blank line between them.
    return "\n\n".join(paragraphs)

In [26]:
# pdf_pages

In [27]:
cleaned_pages = []

for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [29]:
# cleaned_pages

In [30]:
print("Total cleaned pages:", len(cleaned_pages))

Total cleaned pages: 6


In [31]:
print("Cleaned page preview:\n")
print(cleaned_pages[0]["text"])

Cleaned page preview:

Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.

Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA reductase, suppressing endogenou

## 6. Splitting Cleaned Pages into Paragraph

In [33]:
def split_into_paragraph_records(cleaned_pages, min_chars=80):
    paragraph_records = []

    for page in cleaned_pages:
        # Split page text into paragraphs using blank lines.
        paragraphs = page["text"].split("\n\n")

        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            # Remove extra spaces from the beginning and end.
            paragraph = paragraph.strip()

            # Skip very short paragraphs because they are usually headings, page numbers, or noise.
            if len(paragraph) < min_chars:
                continue

            # Store each useful paragraph with basic metadata.
            paragraph_records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return paragraph_records

In [34]:
paragraph_records = split_into_paragraph_records(cleaned_pages)

In [35]:
config.min_chars_per_paragraph

80

In [36]:
print("Total paragraph records:", len(paragraph_records))

Total paragraph records: 9


In [37]:
for record in paragraph_records[:3]:
    print("=" * 80)
    print(f"Page: {record['source_page']} | Paragraph: {record['paragraph_id']} | Characters: {record['char_count']}")
    print(record["text"])

Page: 1 | Paragraph: 1 | Characters: 575
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.
Page: 1 | Paragraph: 2 | Characters: 598
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to monotherapy. Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal wall, reducing cholesterol absorption, while Atorvastatin

## 7. Save extracted and cleaned corpus for auditability

In [38]:
config.processed_data_dir

'/content/pharma_processed_data'

In [39]:
config.processed_data_dir

'/content/pharma_processed_data'

In [40]:
raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

Saved raw pages to: /content/pharma_processed_data/pdf_pages_raw.jsonl
Saved cleaned paragraph corpus to: /content/pharma_processed_data/pharma_paragraph_process.jsonl


## 8. Create Hugging Face Dataset

In [41]:
if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

text_dataset = Dataset.from_list(paragraph_records)

In [42]:
print(text_dataset)

Dataset({
    features: ['text', 'source_page', 'paragraph_id', 'char_count'],
    num_rows: 9
})


In [43]:
print(text_dataset[0])

{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP-activated protein kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory properties. Recent studies also suggest potential anticancer effects through inhibition of the mTOR signaling pathway and suppression of tumor angiogenesis.', 'source_page': 1, 'paragraph_id': 1, 'char_count': 575}


## 9. Train / Eval Split

In [44]:
split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['text', 'source_page', 'paragraph_id', 'char_count'],
        num_rows: 2
    })
})


## 10. Load Tokenizer

In [45]:
config.model_name

'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'

In [46]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [47]:
# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [48]:
tokenizer.eos_token

'</s>'

In [49]:
print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

Tokenizer loaded: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T
Vocab size: 32000
Pad token: </s> | Pad token id: 2
EOS token: </s> | EOS token id: 2


## 11. Tokenization and text packing

In [50]:
def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [51]:
tokenized_datasets = dataset.map(
    tokenize_function,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing text corpus",
)

Tokenizing text corpus:   0%|          | 0/7 [00:00<?, ? examples/s]

Tokenizing text corpus:   0%|          | 0/2 [00:00<?, ? examples/s]

| Parameter                                       | Meaning                                                                                                                               |
| ----------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| `tokenize_function`                             | This function converts each text example into token IDs.                                                                              |
| `batched=True`                                  | The function processes multiple rows at once instead of one row at a time. This makes tokenization faster.                            |
| `remove_columns=datasets["train"].column_names` | After tokenization, the original dataset columns are removed. Only tokenized columns such as `input_ids` and `attention_mask` remain. |
| `desc="Tokenizing text corpus"`                 | This message is shown in the progress bar so we can understand that tokenization is currently running.                                |


In [52]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 7
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 2
    })
})

In [54]:
# tokenized_datasets['train']['input_ids'][0]

In [55]:
def create_training_blocks(tokenized_examples):
    # Join all token IDs from multiple examples into one long list.
    all_input_ids = []
    all_attention_masks = []

    for input_ids in tokenized_examples["input_ids"]:
        all_input_ids.extend(input_ids)

    for attention_mask in tokenized_examples["attention_mask"]:
        all_attention_masks.extend(attention_mask)

    # Calculate how many complete blocks we can create.
    total_tokens = len(all_input_ids)
    usable_tokens = (total_tokens // config.block_size) * config.block_size

    # If we do not have enough tokens to create even one block, return empty data.
    if usable_tokens == 0:
        return {
            "input_ids": [],
            "attention_mask": [],
            "labels": [],
        }

    # Keep only tokens that can fit into complete fixed-size blocks.
    all_input_ids = all_input_ids[:usable_tokens]
    all_attention_masks = all_attention_masks[:usable_tokens]

    # Split the long token list into fixed-size training blocks.
    input_id_blocks = []
    attention_mask_blocks = []

    for start_index in range(0, usable_tokens, config.block_size):
        end_index = start_index + config.block_size

        input_id_blocks.append(all_input_ids[start_index:end_index])
        attention_mask_blocks.append(all_attention_masks[start_index:end_index])

    # For causal language modeling, labels are the same as input IDs.
    # The model uses these labels to learn next-token prediction.
    labels = input_id_blocks.copy()

    return {
        "input_ids": input_id_blocks,
        "attention_mask": attention_mask_blocks,
        "labels": labels,
    }

In [56]:
final_dataset = tokenized_datasets.map(
    create_training_blocks,
    batched=True,
    desc=f"Creating fixed-size training blocks of {config.block_size} tokens",
)

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/7 [00:00<?, ? examples/s]

Creating fixed-size training blocks of 512 tokens:   0%|          | 0/2 [00:00<?, ? examples/s]

In [59]:
sample = final_dataset["train"][0]

In [61]:
# sample

In [62]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

Keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids length: 512
labels length: 512
Decoded sample preview:

<s> Pharma Domain Training Data - Page 5 Page 5 - AI in Drug Discovery and Pharmaceutical R&D; Pharma-domain corpus extension for custom fine-tuning and RAG experimentation. Educational content only; not medical advice. Target identification Artificial intelligence is increasingly used in pharmaceutical research to analyze genomics, transcriptomics, proteomics, disease phenotypes, chemical libraries, and clinical datasets. In target identification, machine learning models can prioritize genes or proteins that may play causal roles in disease biology. These predictions are strengthened when integrated with experimental validation, pathway analysis, human genetics, and disease-relevant biomarkers. Molecular screening In early discovery, deep learning can support virtual screening by predicting protein-ligand binding affinity, molecular properties, toxicity signals,

## 12. Load model with QLoRA-friendly configuration

We use 4-bit quantized loading when CUDA is available.

Why?

- Lower GPU memory usage
- Faster experimentation
- Practical for Colab-style environments
- Common industry approach for parameter-efficient fine-tuning

If CUDA is not available, the notebook falls back to normal loading, but training will be slow on CPU.

In [63]:
use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [64]:
# Clear memory before loading the model.
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [65]:
if use_cuda:
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training

    # Configure 4-bit quantization to reduce GPU memory usage.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Load the base model in 4-bit mode on available GPU devices.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Prepare the quantized model for stable LoRA/QLoRA training.
    base_model = prepare_model_for_kbit_training(base_model)

else:
    # Load the base model normally when GPU is not available.
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to reduce memory usage and avoid training warnings.
base_model.config.use_cache = False

print("Base model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

Base model loaded successfully.


## 13. Apply LoRA adapters

- LoRA trains a small number of adapter parameters instead of updating all base model weights.
- This is cheaper than full fine-tuning and is widely used in real projects.

In [66]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [67]:
model = get_peft_model(base_model, lora_config)

In [68]:
model.print_trainable_parameters()

trainable params: 12,615,680 || all params: 1,112,664,064 || trainable%: 1.1338


## Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?

Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

## 14. Data collator

In [69]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 15. Training arguments

In [70]:
training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_steps=5,
    weight_decay=config.weight_decay,

    # Log training loss at every step for small demo datasets.
    logging_steps=1,
    logging_first_step=True,

    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [71]:
training_args = TrainingArguments(**training_kwargs)

In [73]:
# training_args

## 16. Build Trainer

In [74]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],
    eval_dataset=final_dataset["validation"],
    data_collator=data_collator,
)

print("Trainer is ready.")

Trainer is ready.


## 17. Start Training

In [75]:
train_result = trainer.train()

print("Training completed.")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,2.148971
2,2.148971
3,2.124190


Training completed.


In [76]:
for log in trainer.state.log_history:
    print(log)

{'loss': 2.14897084236145, 'grad_norm': 0.6864576935768127, 'learning_rate': 0.0, 'epoch': 1.0, 'step': 1}
{'loss': 2.1489710807800293, 'grad_norm': 0.6949476599693298, 'learning_rate': 4e-05, 'epoch': 2.0, 'step': 2}
{'loss': 2.124189615249634, 'grad_norm': 0.6684080958366394, 'learning_rate': 8e-05, 'epoch': 3.0, 'step': 3}
{'train_runtime': 21.046, 'train_samples_per_second': 0.855, 'train_steps_per_second': 0.143, 'total_flos': 57901993426944.0, 'train_loss': 2.1407105127970376, 'epoch': 3.0, 'step': 3}


## 18. Save Adapter and Tokenizer

In [77]:
trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

LoRA adapter saved to: /content/pharma_tinyllama_lora_adapter
Saved files:
['adapter_model.safetensors', 'adapter_config.json', 'tokenizer_config.json', 'README.md', 'tokenizer.json']


## 19. Push LoRA adapter to Hugging Face Hub

In [79]:
repo_id = "saadtariq/non-instruction-pharma-finetuning"

model.push_to_hub(
    repo_id,
    private=True,
    token=WRITE_TOKEN
)

tokenizer.push_to_hub(
    repo_id,
    private=True,
    token=WRITE_TOKEN
)

print(f"LoRA adapter pushed to: https://huggingface.co/{repo_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 34.5kB / 50.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.


LoRA adapter pushed to: https://huggingface.co/saadtariq/non-instruction-pharma-finetuning


## 20. Reload base model + LoRA adapter correctly

In [80]:
# Clean old objects to free memory.
del trainer
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [81]:
inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)

if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

In [82]:
if use_cuda:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        ),
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [83]:
inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)

In [85]:
# inference_model.eval()

In [86]:
print("Base model + LoRA adapter loaded successfully for inference.")

Base model + LoRA adapter loaded successfully for inference.


## 21. Inference helper

In [87]:
def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Convert prompt text into token IDs.
    inputs = inference_tokenizer(prompt, return_tensors="pt").to(device)

    # Generate text without calculating gradients because we are doing inference, not training.
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
        )

    # Convert generated token IDs back into readable text.
    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

## 22. Test text continuation

In [88]:
prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

In [89]:
for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

PROMPT:
Metformin is one of the most widely prescribed oral antihyperglycemic agents

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Metformin is one of the most widely prescribed oral antihyperglycemic agents in the world. It is a sulfonylurea class drug, which has been used to treat type 2 diabetes for over 30 years. However, despite its proven efficacy and safety, the mechanism by which it works remains unknown. A recent study from the National Institutes of Health (NIH) reveals that Metformin can directly affect the cellular processes that regulate the production of insulin. This finding could help explain the mechanism of action of Metformin.
Metformin is an orally available compound that is used to treat

PROMPT:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe

MODEL CONTINUATION:


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe, a statin drug that reduces cholesterol and LDL cholesterol, lowers LDL-C by 39% compared to statins alone.
The study was published online in the New England Journal of Medicine.
Atorvastatin is a lipid (fat) lowering drug used for primary prevention of cardiovascular disease, as well as secondary prevention of coronary artery disease, strokes and other types of heart attacks.
Atrial fibrillation is an irregular heartbeat that causes unpredict

PROMPT:
Artificial intelligence is transforming pharmaceutical research by accelerating

MODEL CONTINUATION:
Artificial intelligence is transforming pharmaceutical research by accelerating drug discovery, reducing time-to-market and increasing the efficiency of R&D operations.
Industrial IoT is driving the next wave of innovation in healthcare as a growing number of devices and apps are connected to create smart environments for better patient care.
MedTech IoT is revol

## 23. Optional merge step

In [90]:
merged_model_dir = "/content/pharma_tinyllama_merged_model"
os.makedirs(merged_model_dir, exist_ok=True)

In [91]:
# Reload the base model in float16 for safe merging.
base_model = AutoModelForCausalLM.from_pretrained(
    config.model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [94]:
# Load the trained LoRA adapter on top of the base model.
model_with_adapter = PeftModel.from_pretrained(
    base_model,
    config.adapter_dir
)

In [95]:
# Merge LoRA adapter weights into the base model weights.
merged_model = model_with_adapter.merge_and_unload()

In [96]:
# Save the merged standalone model and tokenizer.

merged_model.save_pretrained(merged_model_dir)

inference_tokenizer.save_pretrained(merged_model_dir)

print(f"Merged model saved to: {merged_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/pharma_tinyllama_merged_model
